> 在线讲义：[第 21 章 · 流水线、上下文、专家并行，以及怎么组合](https://wikieden.github.io/pytorch-inside-out/lessons/21-pp-cp-ep.html) · 本书《PyTorch 从里到外》

# 第 21 章 · 流水线、上下文、专家并行，以及怎么组合

本章回答：按层切模型（PP）为什么会有"气泡"，micro-batch 怎么把气泡缩小？序列太长时 CP 怎么切？MoE 的专家并行（EP）为什么要 all_to_all？五种并行怎么组合，先上哪个？

配套讲义：仓库里 `pytorch-deep-dive/notes/lessons/21-pp-cp-ep.md`（讲解、动画、练习都在讲义里）。对应原文：Ultra-Scale Playbook §Pipeline Parallelism（`ultrascale-playbook.md` 第 1565 行起；气泡公式第 1657–1696 行；1F1B 第 1709 行；Interleaving 第 1767 行；Zero Bubble 与 DualPipe 第 1830 行）、§Context Parallelism（第 1412 行起；Ring Attention 第 1467 行）、§Expert parallelism（第 1892 行）、§5D parallelism（第 1939 行）、§Finding the Best Training Configuration（第 2122 行）；官方文档 `torch.distributed.pipelining`（〔在线〕）；torchtitan（〔在线〕github.com/pytorch/torchtitan）；多机报告 §2.3–§3

**怎么用**：在 `pytorch-deep-dive/solutions/chapter-code/` 下用 `uv run jupyter lab` 打开（或任何装了 torch 的环境）。从上往下，一格一格按 **Shift + Enter** 运行。文字格是讲解，代码格是对应的写法；数字随便改，改完再按一次 Shift + Enter 看结果怎么变。Mac 上能跑：计时类的格子在 Mac 上用 MPS，分布式的格子在本机起 2 个 Gloo 进程。

In [ ]:
print("准备好了")

## 讲义 ↔ 代码

先看这张表：讲义里的每一步，对应下面哪一个函数、跑出来是什么。

| 讲义 | 公式 / 说法 | 写法 | 跑出来 |
|---|---|---|---|
| 第一步 单 batch 气泡 | p − 1 | `bubble_ratio(4)` | 3 |
| 第二步 m 个小批 | (p − 1)/m | `bubble_ratio(4, 8)`、`(4, 32)` | 0.375、0.094 |
| 第二步 逐格模拟 | 前向 1 格、反向 2 格 | `simulate_afab(4, 8)` | 总长 33、有效 24、比值 0.375 |

## 1. 先把要用的东西定义好

这几格只是定义（函数、类、常量），运行后没有输出也正常。用到的前几章的函数也一起抄进来了，不需要本地文件。

**函数 `bubble_ratio`**（来自 `ch21_pp_cp_ep.py`）：气泡 / 有效时间 = (p − 1) / m（Playbook 第 1693 行）。

In [ ]:
def bubble_ratio(p: int, m: int = 1) -> float:
    """气泡 / 有效时间 = (p − 1) / m（Playbook 第 1693 行）。"""
    return (p - 1) / m

**函数 `simulate_afab`**（来自 `ch21_pp_cp_ep.py`）：逐格模拟 AFAB 调度：返回 (总时长, 每卡有效时长, 卡 0 同时存的激活份数峰值)。

In [ ]:
def simulate_afab(p: int, m: int, tf: int = 1, tb: int = 2):
    """逐格模拟 AFAB 调度：返回 (总时长, 每卡有效时长, 卡 0 同时存的激活份数峰值)。"""
    f_end = [[0] * m for _ in range(p)]
    t = 0
    for s in range(p):                      # 前向：第 s 段第 i 个小批，要等上一段算完
        t = 0
        for i in range(m):
            start = max(t, f_end[s - 1][i] if s else 0)
            f_end[s][i] = start + tf
            t = f_end[s][i]
    b_end = [[0] * m for _ in range(p)]
    for s in reversed(range(p)):
        t = f_end[s][m - 1]
        for i in range(m):
            dep = f_end[s][i] if s == p - 1 else b_end[s + 1][i]
            start = max(t, dep)
            b_end[s][i] = start + tb
            t = b_end[s][i]
    total = max(max(r) for r in b_end)
    return total, m * (tf + tb), m

## 2. 一步一步跑

每格对应讲义里的一步，打印的文字说明了在算什么。

In [ ]:
print("\n第一步 只送一个 batch：p = 4 →", bubble_ratio(4), "倍")

In [ ]:
print("\n第二步 p = 4、m = 8 →", bubble_ratio(4, 8), "；m = 32 →", round(bubble_ratio(4, 32), 3))
total, useful, peak = simulate_afab(4, 8)
print("逐格模拟 AFAB p=4、m=8：总长", total, "有效", useful, "气泡/有效", round((total - useful) / useful, 3), "卡 0 激活峰值", peak)

## 3. 代码里才看得到的

1. **模拟和公式一致**：只要 t_b 是常数倍 t_f，比值就是 (p − 1)/m。
2. **AFAB 卡 0 同时存 m 份激活**：这就是 1F1B 要解决的问题（动画里能切换看）。

## 4. 自己改一改

- `simulate_afab(6, 4)` 和公式对一对。
- 把 tb 改成 1，看比值变不变。

讲义末尾的练习题可以在下面的空格子里用上面的函数验算——先自己在纸上算，再来核对。

In [ ]:
# 在这里写你的代码

## 口诀

> 按层切成流水线，段间传激活；一个 batch 气泡大，p − 1 倍在空等；
> 切成 m 个小批，气泡降到 (p − 1) / m；1F1B 前反交替，激活不随 m 涨；
> 序列太长 CP 绕圈传 KV，专家分卡 all_to_all；最勤的通信放机内，最疏的放外层。